# Assignment 2, Part 2: Logistic Regression

Logistic regression trained on the XOR dataset using batch gradient descent. The feature map adds an interaction term, and the model uses sigmoid activation and binary cross-entropy loss. All calculations are implemented manually using Python.

## XOR dataset and initial weights
The feature map is $\psi(x)=(x_1,x_2,x_1x_2)$. A bias feature $x_0=1$ is included.

In [1]:
import math

# Features: [bias, x1, x2, x1*x2]
X = [
    [1, 0, 0, 0],
    [1, 0, 1, 0],
    [1, 1, 0, 0],
    [1, 1, 1, 1]
]
t = [0, 1, 1, 0]

initial_weights = [-0.6, 2.0, 1.2, -2.8]
learning_rate = 0.1
max_iterations = 100000
loss_threshold = 0.01

print('Training examples:', len(X))
print('Initial weights:', initial_weights)
print('Learning rate:', learning_rate)

Training examples: 4
Initial weights: [-0.6, 2.0, 1.2, -2.8]
Learning rate: 0.1


## Logistic regression model
Calculate $z=w^Tx$ and $y=\sigma(z)=1/(1+e^{-z})$. Binary cross-entropy is $L=-t\ln(y)-(1-t)\ln(1-y)$.

In [2]:
def sigmoid(z):
    return 1 / (1 + math.exp(-z))


def predict(X, weights):
    z_values = []
    y_values = []
    for x in X:
        z = sum(weights[j] * x[j] for j in range(len(weights)))
        z_values.append(z)
        y_values.append(sigmoid(z))
    return z_values, y_values


def cross_entropy(y_values, targets):
    losses = []
    for y, target in zip(y_values, targets):
        # Clamp probabilities to avoid taking log(0).
        y = max(1e-15, min(1 - 1e-15, y))
        loss = -(target * math.log(y) + (1 - target) * math.log(1 - y))
        losses.append(loss)
    return losses, sum(losses) / len(losses)

## Batch gradient descent
The gradient for weight $w_j$ is $\frac{1}{N}\sum_i(y^{(i)}-t^{(i)})x_j^{(i)}$. Update all weights together using $w_j\leftarrow w_j-\alpha\,\mathrm{gradient}_j$.

In [3]:
def calculate_gradients(X, targets, y_values):
    N = len(X)
    gradients = [0.0] * len(X[0])
    for i in range(N):
        error = y_values[i] - targets[i]
        for j in range(len(gradients)):
            gradients[j] += error * X[i][j]
    return [gradient / N for gradient in gradients]


def train(X, targets, initial_weights, lr):
    weights = initial_weights.copy()
    history = []
    for iteration in range(1, max_iterations + 1):
        _, predictions = predict(X, weights)
        gradients = calculate_gradients(X, targets, predictions)
        weights = [weights[j] - lr * gradients[j] for j in range(len(weights))]

        z_values, y_values = predict(X, weights)
        losses, average_loss = cross_entropy(y_values, targets)
        history.append({
            'iteration': iteration,
            'z': z_values,
            'y': y_values,
            'losses': losses,
            'average_loss': average_loss
        })
        if average_loss < loss_threshold:
            break
    return weights, history


final_weights, history = train(X, t, initial_weights, learning_rate)
print('Total iterations:', len(history))
print(f'Final average cross-entropy loss: {history[-1]["average_loss"]:.8f}')

Total iterations: 25356
Final average cross-entropy loss: 0.00999966


## Result 1: last 10 weight updates
The values below are calculated after each weight update. Each row shows the linear output $z$, sigmoid probability $y$, target, and individual cross-entropy loss.

In [4]:
for result in history[-10:]:
    print(f"\nIteration: {result['iteration']}")
    print(f"{'Input':<10} {'z':>11} {'y':>11} {'Target':>8} {'Loss':>11}")
    for i, x in enumerate(X):
        pair = f'({x[1]}, {x[2]})'
        print(f"{pair:<10} {result['z'][i]:>11.6f} {result['y'][i]:>11.6f} "
              f"{t[i]:>8d} {result['losses'][i]:>11.6f}")
    print(f"Average loss: {result['average_loss']:.8f}")


Iteration: 25347
Input                z           y   Target        Loss
(0, 0)       -4.230493    0.014337        0    0.014440
(0, 1)        4.641663    0.990450        1    0.009595
(1, 0)        4.641663    0.990450        1    0.009595
(1, 1)       -5.051155    0.006361        0    0.006382
Average loss: 0.01000323

Iteration: 25348
Input                z           y   Target        Loss
(0, 0)       -4.230533    0.014336        0    0.014440
(0, 1)        4.641703    0.990451        1    0.009595
(1, 0)        4.641703    0.990451        1    0.009595
(1, 1)       -5.051195    0.006361        0    0.006381
Average loss: 0.01000283

Iteration: 25349
Input                z           y   Target        Loss
(0, 0)       -4.230573    0.014336        0    0.014439
(0, 1)        4.641743    0.990451        1    0.009595
(1, 0)        4.641743    0.990451        1    0.009595
(1, 1)       -5.051235    0.006361        0    0.006381
Average loss: 0.01000243

Iteration: 25350
Input        

## Result 2: final weights and loss

In [5]:
for j, weight in enumerate(final_weights):
    print(f'w{j} = {weight:.6f}')
print(f'Total iterations: {len(history)}')
print(f'Final average cross-entropy loss: {history[-1]["average_loss"]:.8f}')

w0 = -4.230853
w1 = 8.872874
w2 = 8.872874
w3 = -18.566406
Total iterations: 25356
Final average cross-entropy loss: 0.00999966


## Result 3: final XOR predictions
A predicted probability of at least 0.5 is classified as 1; otherwise it is classified as 0.

In [6]:
z_values, y_values = predict(X, final_weights)
correct = 0
print(f"{'Input':<10} {'Target':>8} {'Probability':>13} {'Prediction':>12}")
for i, x in enumerate(X):
    label = 1 if y_values[i] >= 0.5 else 0
    correct += (label == t[i])
    pair = f'({x[1]}, {x[2]})'
    print(f'{pair:<10} {t[i]:>8} {y_values[i]:>13.6f} {label:>12}')
accuracy = correct / len(X)
print(f'Accuracy: {accuracy:.2%}')
assert correct == len(X), 'Not all XOR examples were classified correctly.' 

Input        Target   Probability   Prediction
(0, 0)            0      0.014332            0
(0, 1)            1      0.990454            1
(1, 0)            1      0.990454            1
(1, 1)            0      0.006359            0
Accuracy: 100.00%
